In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets  import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection  import train_test_split

In [2]:
x,y=make_classification(
    n_features=10,
    n_samples=1000,
    n_informative=4,
    n_redundant=6,
    n_repeated=0,
    n_classes=2,
    weights=[0.9,0.1],
    random_state=42
)

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.25,random_state=42)

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# method 1: manually adjust parameter

In [4]:
model=DecisionTreeClassifier(criterion='entropy',max_depth=10)
model.fit(x_train,y_train)

y_pred=model.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95       221
           1       0.62      0.52      0.57        29

    accuracy                           0.91       250
   macro avg       0.78      0.74      0.76       250
weighted avg       0.90      0.91      0.90       250



# method 2: cross validation score

In [5]:
from sklearn.model_selection import cross_val_score

cross_val_score(DecisionTreeClassifier(criterion='gini',max_depth=5),x,y,cv=5)

array([0.945, 0.945, 0.94 , 0.975, 0.915])

In [6]:
cross_val_score(DecisionTreeClassifier(criterion='gini',max_depth=10),x,y,cv=5)

array([0.935, 0.94 , 0.95 , 0.95 , 0.915])

In [7]:
cross_val_score(DecisionTreeClassifier(criterion='entropy',max_depth=5),x,y,cv=5)

array([0.945, 0.89 , 0.94 , 0.95 , 0.915])

# method 3: Gride search and random search

## using for loop

In [11]:
criterion=['gini','entropy']
max_depth=[5,10,15]

avg_score={}

for c in criterion:
    for d in max_depth:
        clf=DecisionTreeClassifier(criterion=c,max_depth=d)
        score_list=cross_val_score(clf,x,y,cv=5)
        avg_score[c+'_'+str(d)]=round(np.average(score_list),4)

avg_score

{'gini_5': np.float64(0.945),
 'gini_10': np.float64(0.944),
 'gini_15': np.float64(0.937),
 'entropy_5': np.float64(0.927),
 'entropy_10': np.float64(0.942),
 'entropy_15': np.float64(0.934)}

## Gride Search method

In [12]:
from sklearn.model_selection import GridSearchCV

clf=GridSearchCV(DecisionTreeClassifier(),
                 {'criterion':['gini','entropy']
                 ,'max_depth':[5,10,15]},
                 cv=5,
                 return_train_score=False)

In [14]:
clf.fit(x,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'criterion': ['gini', 'entropy'], 'max_depth': [5, 10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : 

In [15]:
clf.cv_results_

{'mean_fit_time': array([0.0062603 , 0.01127172, 0.01155949, 0.00939288, 0.01012974,
        0.00981665]),
 'std_fit_time': array([0.00047489, 0.00201399, 0.00212908, 0.00259577, 0.00091768,
        0.00112802]),
 'mean_score_time': array([0.00124907, 0.00222669, 0.00248585, 0.00141912, 0.00113912,
        0.00101185]),
 'std_score_time': array([0.0002908 , 0.0008271 , 0.00101825, 0.00065475, 0.00015603,
        0.00010901]),
 'param_criterion': masked_array(data=['gini', 'gini', 'gini', 'entropy', 'entropy',
                    'entropy'],
              mask=[False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'param_max_depth': masked_array(data=[5, 10, 15, 5, 10, 15],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'params': [{'criterion': 'gini', 'max_depth': 5},
  {'criterion': 'gini', 'max_depth': 10},
  {'criterion': 'gini', 'max_depth': 15},
  {'criterion': 'entropy', 'max_depth': 5

In [ ]:
import pandas as pd

df=pd.DataFrame(clf.cv_results_)# 6 combination
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.006260,0.000475,0.001249,0.000291,gini,5,"{'criterion': 'gini', 'max_depth': 5}",0.940,0.945,0.940,0.965,0.915,0.941,0.015937,2
1,0.011272,0.002014,0.002227,0.000827,gini,10,"{'criterion': 'gini', 'max_depth': 10}",0.950,0.940,0.950,0.960,0.915,0.943,0.015362,1
2,0.011559,0.002129,0.002486,0.001018,gini,15,"{'criterion': 'gini', 'max_depth': 15}",0.935,0.935,0.945,0.960,0.905,0.936,0.018000,5
3,0.009393,0.002596,0.001419,0.000655,entropy,5,"{'criterion': 'entropy', 'max_depth': 5}",0.945,0.890,0.940,0.950,0.915,0.928,0.022494,6
4,0.010130,0.000918,0.001139,0.000156,entropy,10,"{'criterion': 'entropy', 'max_depth': 10}",0.955,0.945,0.930,0.935,0.925,0.938,0.010770,3
5,0.009817,0.001128,0.001012,0.000109,entropy,15,"{'criterion': 'entropy', 'max_depth': 15}",0.960,0.940,0.935,0.935,0.915,0.937,0.014353,4


In [17]:
df[['param_criterion','param_max_depth','mean_test_score']]

,param_criterion,param_max_depth,mean_test_score
0,gini,5,0.941
1,gini,10,0.943
2,gini,15,0.936
3,entropy,5,0.928
4,entropy,10,0.938
5,entropy,15,0.937


In [18]:
clf.best_params_ # to get the best parameter from the combination

{'criterion': 'gini', 'max_depth': 10}

### when we need to check parameter for multiple model at some time

In [23]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

model_params = {
    'Decision_Tree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy'],
            'max_depth': [5, 10, 15]
        }
    },

    'SVM': {
        'model': SVC(gamma='auto'),
        'params': {
            'C': [1, 10, 20],
            'kernel': ['rbf', 'linear']
        }
    },

    'XGBoost': {
        'model': XGBClassifier(),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.2]
        }
    }
}

In [24]:
score=[]

for key,val in model_params.items():
    clf=GridSearchCV(val['model'],val['params'],cv=5,return_train_score=False)
    clf.fit(x,y)
    score.append({
        'model':key,
        'best_score':clf.best_score_,
        'best_params':clf.best_params_
    })
    clf.best_params_

In [27]:
df = pd.DataFrame(score)
df

,model,best_score,best_params
0,Decision_Tree,0.943,"{'criterion': 'gini', 'max_depth': 5}"
1,SVM,0.964,"{'C': 20, 'kernel': 'rbf'}"
2,XGBoost,0.958,"{'learning_rate': 0.1, 'max_depth': 7, 'n_esti..."


## random search

In [28]:
from sklearn.model_selection import RandomizedSearchCV

In [29]:
score=[]

for key,val in model_params.items():
    clf=RandomizedSearchCV(val['model'],val['params'],cv=5,return_train_score=False)
    clf.fit(x,y)
    score.append({
        'model':key,
        'best_score':clf.best_score_,
        'best_params':clf.best_params_
    })
    clf.best_params_

C:\Users\Jeevan kumar\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\Jeevan kumar\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
